# Piloto de harmonizacion v0.4 sobre CMAPSS

Reescritura del piloto siguiendo `CLAUDE.md` (sec. 6, 9, 10, 11, 12, 13). El objetivo es validar end-to-end el contrato tensorial channel-independent, las mascaras, la normalizacion instance, los checks anti-leakage y los manifests, **antes** de escalar a `05_harmonization_full.ipynb`.

Configuracion del MVP (fija para esta iteracion):

- `window_size = 512`
- `stride = 256`
- `patch_size = 16`
- `n_patches = 32`
- `mask_ratio = 0.3`
- `shard_maxcount = 5000`
- `pipeline_version = "0.4"`
- `tail_policy = "pad"` (alineado con audit v2.3)

Contrato tensorial (CLAUDE.md sec. 9):

- serie por unidad: `(T, C)`
- ventana: `(W, C)`
- patches en disco por sample: `(C, N, P)` float32
- batch del DataLoader: `(B, C, N, P)`
- reshape interno esperado por el modelo: `(B*C, N, P)`
- patch embedding futuro: `Linear(P, d_model)`, **nunca** `Linear(P*C, d_model)`

Por sample se persisten ademas:

- `valid_time_mask`: `(W,)` bool, marca timesteps reales (no padding)
- `valid_patch_mask`: `(C, N)` bool, True si el patch tiene al menos una muestra real
- `mean`: `(C,)` float32, estadistico calculado antes de normalizar, ignorando padding
- `std`: `(C,)` float32, idem
- `target` separado de las features (no se guarda como canal)
- `meta` con `dataset, role, subset_id, split_original, unit_id, unit_global_id, window_start, window_end`

Para CMAPSS:

- Role: `TRANSFER_TARGET`, evaluation_tier `primary`.
- Splits: respetar los originales que entrega `phmd` (`train`/`val`/`test`).
- Target: `rul` con `target_policy = ultimo_valor_valido`. **No transformar** ni clampear. Guardar `target_raw` y `target_warning = rul_negative_values` cuando `rul_min < 0`.
- `sampling_rate_hz = null`: CMAPSS no expone frecuencia fiable. `window_time_seconds = null`. W=512 son muestras, no duracion fisica.

Cambios v0.3 -> v0.4:

- `tail_policy = "pad"` operacional, leido del audit v2.3 y aplicado tambien en `ventanas_de_unidad`.
- `estimated_n_shards` se LEE del CSV del audit (no se recalcula).
- Reentrancia segura por `pipeline_config_hash` (no solo `done.flag`). `WIPE_V1` renombrado a `FORCE_REPROCESS`.
- Asserts duros tras el run: piloto debe coincidir byte a byte con el audit en metricas comparables.
- `pipeline_code_version` se calcula con prioridad env var -> git rev-parse (read-only).
- Manifest y `processed_summary.csv` con columnas de trazabilidad completas: `audit_version`, `tail_policy`, hashes, metricas densas, `source_rows_*`.

Politica de output:

- `FORCE_REPROCESS = True` (default del piloto): borra `processed/CMAPSS/` completo y reprocesa.
- `WIPE_V1` se conserva como alias retrocompatible en codigo, pero la documentacion usa `FORCE_REPROCESS`.
- Output limpio en `processed/CMAPSS/` con `pipeline_version: "0.4"` en el manifest. **No** se crea subcarpeta `v2/`.
- Sin commits automaticos.


## Setup

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh

## 1. Configuracion

Constantes del piloto. `FORCE_REPROCESS=True` autoriza explicitamente el borrado de `processed/CMAPSS/` y un reproceso limpio (default del piloto v0.4). `WIPE_V1` se conserva como alias retrocompatible en codigo. `ALLOW_CHANNEL_DISCREPANCY=False` hace que el piloto pare si el numero de canales detectado no coincide con el del audit (ver seccion 5).


In [ ]:
from pathlib import Path

DATASET           = 'CMAPSS'
PIPELINE_VERSION  = '0.4'

# Politica de cola del ventaneo (debe coincidir con la del audit v2.3).
TAIL_POLICY       = 'pad'

# Hiperparametros fijos del MVP (CLAUDE.md sec.12)
WINDOW_SIZE = 512
STRIDE      = 256
PATCH_SIZE  = 16
N_PATCHES   = WINDOW_SIZE // PATCH_SIZE
MASK_RATIO  = 0.3

# Persistencia
SHARD_MAXCOUNT = 5000  # ventanas por shard .tar (alineado con audit)

# Flags operativos
FORCE_REPROCESS = True   # True: borra processed/CMAPSS/ y reprocesa siempre.
                         # False: salta si el manifest existente coincide con
                         # la configuracion actual (pipeline_config_hash).
# Alias retro-compat (nombre historico):
WIPE_V1 = FORCE_REPROCESS
ALLOW_CHANNEL_DISCREPANCY = False    # si False: para si n_canales detectado != audit

# Limpieza NaN
MAX_NAN_PCT_UNIDAD = 30.0

# Rutas en Drive
DRIVE_ROOT     = Path('/content/drive/MyDrive/fm_fl_phmd')
RAW_DIR        = DRIVE_ROOT / 'raw'
AUDIT_DIR      = DRIVE_ROOT / 'audit'
PROCESSED_DIR  = DRIVE_ROOT / 'processed' / DATASET
AUDIT_CACHE    = Path('/content/_audit_cache')

# Rutas en el repositorio
REPO            = Path('/content/fm_fl_phmd')
AUDIT_CSV       = REPO / 'results' / 'audit' / 'audit_report.csv'
AUDIT_SUMMARY   = REPO / 'results' / 'audit' / 'audit_summary.json'
PROCESSED_SUMMARY = REPO / 'results' / 'processed_summary.csv'

print(f'Dataset           : {DATASET}')
print(f'PIPELINE_VERSION  : {PIPELINE_VERSION}')
print(f'TAIL_POLICY       : {TAIL_POLICY}')
print(f'W={WINDOW_SIZE} S={STRIDE} P={PATCH_SIZE} N={N_PATCHES} mask={MASK_RATIO}')
print(f'Output            : {PROCESSED_DIR}')
print(f'FORCE_REPROCESS   : {FORCE_REPROCESS}')
print(f'ALLOW_CHANNEL_DISCREPANCY: {ALLOW_CHANNEL_DISCREPANCY}')


## 2. Preflight contra `audit_report.csv`

Antes de tocar nada, leemos lo que el audit dice de CMAPSS y verificamos coherencia. Si el audit y el loader entran en conflicto serio, paramos. Registramos los warnings conocidos para incluirlos en el manifest final.

In [ ]:
import json
import pandas as pd

df_audit = pd.read_csv(AUDIT_CSV)
fila = df_audit[df_audit['dataset'] == DATASET]
assert len(fila) == 1, f'{DATASET} no encontrado o duplicado en {AUDIT_CSV}'
fila = fila.iloc[0]

# Preflight v0.4 contra audit v2.3.
audit_version     = str(fila['audit_version'])
audit_tail_policy = str(fila['tail_policy'])
audit_role        = fila['role']
audit_tier        = fila['evaluation_tier']
audit_target      = fila['target_col']
audit_cands_raw   = fila['target_candidates'] or ''
audit_n_canales   = int(fila['n_canales']) if pd.notna(fila['n_canales']) else None
audit_n_windows   = int(fila['n_windows']) if pd.notna(fila['n_windows']) else None
audit_pad_ratio   = float(fila['padding_ratio']) if pd.notna(fila['padding_ratio']) else None
audit_rul_min     = float(fila['rul_min']) if pd.notna(fila['rul_min']) else None
audit_rul_max     = float(fila['rul_max']) if pd.notna(fila['rul_max']) else None
audit_n_channel_patches_valid = int(fila['n_channel_patches']) if pd.notna(fila['n_channel_patches']) else None
audit_n_temporal_patches      = int(fila['n_temporal_patches']) if pd.notna(fila['n_temporal_patches']) else None
audit_n_filas_total           = int(fila['n_filas_total']) if pd.notna(fila['n_filas_total']) else None
# Nuevas lecturas v2.3:
audit_n_dense_temporal_patches = int(fila['n_dense_temporal_patches']) if pd.notna(fila['n_dense_temporal_patches']) else None
audit_n_dense_channel_patches  = int(fila['n_dense_channel_patches'])  if pd.notna(fila['n_dense_channel_patches'])  else None
audit_invalid_patch_ratio      = float(fila['invalid_patch_ratio'])    if pd.notna(fila['invalid_patch_ratio'])    else None
audit_dense_vs_valid_ratio     = float(fila['dense_vs_valid_ratio'])   if pd.notna(fila['dense_vs_valid_ratio'])   else None
# estimated_n_shards LEIDO del CSV (no recalculado).
audit_estimated_n_shards = int(fila['estimated_n_shards']) if pd.notna(fila['estimated_n_shards']) else None

print(f'Audit dice de {DATASET}:')
print(f'  audit_version     : {audit_version}')
print(f'  audit_tail_policy : {audit_tail_policy}')
print(f'  role              : {audit_role}')
print(f'  evaluation_tier   : {audit_tier}')
print(f'  target_col        : {audit_target}')
print(f'  target_candidates : {audit_cands_raw}')
print(f'  n_canales         : {audit_n_canales}')
print(f'  n_windows         : {audit_n_windows}')
print(f'  padding_ratio     : {audit_pad_ratio}')
print(f'  n_temporal_patches (valid)  : {audit_n_temporal_patches}')
print(f'  n_channel_patches  (valid)  : {audit_n_channel_patches_valid}')
print(f'  n_dense_temporal_patches    : {audit_n_dense_temporal_patches}')
print(f'  n_dense_channel_patches     : {audit_n_dense_channel_patches}')
print(f'  invalid_patch_ratio         : {audit_invalid_patch_ratio}')
print(f'  dense_vs_valid_ratio        : {audit_dense_vs_valid_ratio}')
print(f'  estimated_n_shards          : {audit_estimated_n_shards}')
print(f'  n_filas_total               : {audit_n_filas_total}')
print(f'  rul range                   : [{audit_rul_min}, {audit_rul_max}]')

# Aserciones duras del preflight v0.4:
assert audit_version == '2.3', (
    f'Piloto v0.4 requiere audit v2.3, audit actual={audit_version}. '
    f'Re-ejecutar 02_dataset_audit.ipynb con AUDIT_VERSION=2.3.'
)
assert audit_tail_policy == 'pad', (
    f'Piloto v0.4 requiere tail_policy=pad en el audit, encontrado={audit_tail_policy}'
)
assert TAIL_POLICY == 'pad', f'Piloto debe usar tail_policy=pad, no {TAIL_POLICY}'
assert audit_role == 'TRANSFER_TARGET', f'CMAPSS debe ser TRANSFER_TARGET, no {audit_role}'
assert audit_tier == 'primary', f'CMAPSS debe ser evaluation_tier=primary, no {audit_tier}'
assert 'rul' in audit_cands_raw.split(';'), f'target_candidates debe incluir rul: {audit_cands_raw}'
assert audit_estimated_n_shards is not None, 'audit_report.csv no tiene estimated_n_shards para CMAPSS'

warnings_conocidos = []
if audit_pad_ratio is not None and audit_pad_ratio > 0.15:
    warnings_conocidos.append({'tipo': 'padding_ratio_alto', 'valor': audit_pad_ratio})
if audit_rul_min is not None and audit_rul_min < 0:
    warnings_conocidos.append({'tipo': 'rul_negative_values', 'valor': audit_rul_min})
if audit_role == 'TRANSFER_TARGET' and audit_pad_ratio and audit_pad_ratio > 0.5:
    warnings_conocidos.append({'tipo': 'tt_padding_alto', 'valor': audit_pad_ratio})

print()
print(f'Warnings conocidos del audit: {warnings_conocidos}')
print(f'Preflight v0.4: OK (audit v{audit_version}, tail_policy={audit_tail_policy})')


## 3. Borrado autorizado de `processed/CMAPSS/`

`FORCE_REPROCESS=True` autoriza el borrado de la carpeta del dataset y el reproceso limpio. Solo se toca `processed/CMAPSS/`. No se toca `raw/`, ni `checkpoints/`, ni `results/audit/`, ni otros datasets dentro de `processed/`.

Si `FORCE_REPROCESS=False`, el piloto valida el manifest existente y el `done.flag` antes de saltar: si el manifest coincide con la configuracion basica esperada y el `done.flag` esta presente, aborta limpio con `SystemExit`. Si falta `done.flag` o el manifest no coincide, trata el artefacto como stale y reprocesa.


In [ ]:
import json
import shutil


def manifest_matches_config(manifest_path, expected_config):
    """Devuelve (ok, motivo). True solo si todos los campos esperados estan
    presentes y coinciden con los del manifest existente.
    """
    if not manifest_path.exists():
        return False, 'no existe manifest.json'
    try:
        m = json.loads(manifest_path.read_text())
    except Exception as e:
        return False, f'manifest no se puede leer: {e}'
    for k, v in expected_config.items():
        if m.get(k) != v:
            return False, f'campo {k} difiere: manifest={m.get(k)!r} expected={v!r}'
    return True, 'match'


# Configuracion basica para validacion temprana. Es la huella minima de un
# artefacto procesado anteriormente. El hash completo (con normalization,
# nan, target, signal_cols_hash, metadata_cols_hash) se calcula en cell 35
# y se compara contra el manifest existente cuando se quiera reentrancia
# completa (uso previsto en 05_harmonization_full).
expected_basic_config = {
    'dataset':          DATASET,
    'audit_version':    audit_version,        # leido del CSV en cell 6
    'pipeline_version': PIPELINE_VERSION,
    'tail_policy':      TAIL_POLICY,
    'window_size':      WINDOW_SIZE,
    'stride':           STRIDE,
    'patch_size':       PATCH_SIZE,
}

manifest_existente = PROCESSED_DIR / 'manifest.json'
done_flag          = PROCESSED_DIR / 'done.flag'

# Flag de skip global. Se setea a True solo si TODOS los criterios coinciden.
SKIP_PROCESSING = False

if FORCE_REPROCESS:
    if PROCESSED_DIR.exists():
        existentes = sorted(PROCESSED_DIR.iterdir())
        print(f'FORCE_REPROCESS=True. Borrando {len(existentes)} entradas en {PROCESSED_DIR}:')
        for p in existentes[:20]:
            print(f'  - {p.name}')
        if len(existentes) > 20:
            print(f'  ... ({len(existentes) - 20} mas)')
        shutil.rmtree(PROCESSED_DIR)
        print(f'Borrada: {PROCESSED_DIR}')
    else:
        print(f'{PROCESSED_DIR} no existia.')
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Carpeta limpia creada: {PROCESSED_DIR}')
    print()
    print(f'expected_basic_config (lo que escribira este run en manifest.json):')
    for k, v in expected_basic_config.items():
        print(f'  {k:20s} {v!r}')
else:
    # Reentrancia minima del piloto: valida config basica + done.flag.
    # Si done.flag falta, el artefacto se considera incompleto (stale)
    # incluso si manifest.json existe y coincide.
    print(f'FORCE_REPROCESS=False. Validando manifest + done.flag contra config basica...')
    ok_manifest, motivo_manifest = manifest_matches_config(manifest_existente, expected_basic_config)
    ok_done = done_flag.exists()
    if ok_manifest and ok_done:
        SKIP_PROCESSING = True
        print(f'manifest.json coincide y done.flag presente: {motivo_manifest}.')
        print(f'Re-ejecutar con FORCE_REPROCESS=True para forzar reproceso.')
        print(f'')
        print(f'Para evitar reescribir artefactos validos, ABORTAMOS aqui.')
        # SystemExit corta limpio el run sin traceback de error en Colab.
        raise SystemExit('SKIP_PROCESSING: manifest existente coincide y done.flag presente.')
    else:
        motivo_skip = []
        if not ok_manifest: motivo_skip.append(f'manifest: {motivo_manifest}')
        if not ok_done:     motivo_skip.append('done.flag no existe (proceso anterior incompleto)')
        print(f'No se puede saltar: ' + '; '.join(motivo_skip))
        print(f'Stale: reprocesamos.')
        if PROCESSED_DIR.exists():
            shutil.rmtree(PROCESSED_DIR)
        PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
        print(f'Carpeta limpia creada: {PROCESSED_DIR}')

# NOTA: La reentrancia completa por pipeline_config_hash (que incluye
# normalization_policy, nan_policy, target_col, subset_id_col,
# signal_cols_hash, metadata_cols_hash) se utiliza en
# 05_harmonization_full. En el piloto, FORCE_REPROCESS=True es la ruta
# recomendada para garantizar reproceso limpio.


## 4. Cargar CMAPSS via phmd

Misma estrategia que el audit: copia del `.zip` desde Drive a la cache local rapida (`/content/_audit_cache/datasets/`) para que la descompresion ocurra en SSD, no en Drive. Tras la carga limpiamos `/root/.phmd` y la cache local. Tomamos el primer task (`rul`) y primer fold, que en CMAPSS devuelve los 4 FDs concatenados (FD001-FD004) en un unico DataFrame por split, con la columna `FD` distinguiendo cada subset.

In [ ]:
import shutil
import sys
import zipfile
from pathlib import Path
from phmd import datasets as phmd_datasets

PHMD_HOME_CACHE = '/root/.phmd'


def _inyectar_zipfile_en_phmd():
    """Workaround heredado del audit v2.2.1 (caso PRONOSTIA): inyecta
    zipfile en cualquier submodulo phmd ya cargado para evitar NameError."""
    for nombre_mod, mod in list(sys.modules.items()):
        if nombre_mod.startswith('phmd') and mod is not None:
            try:
                setattr(mod, 'zipfile', zipfile)
            except Exception:
                pass


_inyectar_zipfile_en_phmd()

# Copiamos el zip directamente a /root/.phmd/datasets/ (donde phmd lo busca
# realmente). v0.2 copiaba a /content/_audit_cache pero phmd ignora ese
# cache_dir en algunos readers; provoco "Dataset not found" en la corrida
# del 2026-05-19 dentro del piloto disparado por la celda de encadenado.
phmd_cache  = Path(PHMD_HOME_CACHE) / 'datasets'
phmd_cache.mkdir(parents=True, exist_ok=True)
drive_files = RAW_DIR / 'datasets'

n_lc = DATASET.lower()
ficheros_copiados_pilot = []
for f in drive_files.iterdir():
    if not f.is_file():
        continue
    if (f.stem.lower() == n_lc
            or f.stem.lower().startswith(n_lc + '_')
            or f.name.lower().startswith(n_lc + '.')):
        destino = phmd_cache / f.name
        try:
            tamano_origen = f.stat().st_size
        except Exception:
            tamano_origen = -1
        recopiar = (not destino.exists()) or (
            tamano_origen > 0 and destino.stat().st_size != tamano_origen
        )
        if recopiar:
            shutil.copy(f, destino)
        if destino.stat().st_size == 0:
            raise IOError(f'Copia incompleta a {destino} (0 bytes). '
                          f'Drive probablemente se desmonto. Reintenta.')
        ficheros_copiados_pilot.append(destino)
        print(f'  copiado a /root/.phmd/datasets/: {destino.name}')

# Carga con reintento si phmd dispara NameError de zipfile.
try:
    ds   = phmd_datasets.Dataset(DATASET, cache_dir=PHMD_HOME_CACHE)
except NameError as ne:
    if 'zipfile' in str(ne).lower():
        _inyectar_zipfile_en_phmd()
        ds = phmd_datasets.Dataset(DATASET, cache_dir=PHMD_HOME_CACHE)
    else:
        raise

task = ds.tasks[0]
fold = task[0]

if isinstance(fold, dict):
    splits = {k: v for k, v in fold.items() if v is not None}
elif isinstance(fold, (tuple, list)):
    nombres = ['train', 'val', 'test'][:len(fold)]
    splits  = {n: v for n, v in zip(nombres, fold) if v is not None}
else:
    splits = {'train': fold}

print()
for nombre, df in splits.items():
    print(f'  split {nombre:5s}: {len(df):>8,d} filas, {df.shape[1]} columnas')


## 5. Deteccion de columnas y discrepancia con el audit

Desde audit v2.3, `FD` se trata como `subset_id_col` tambien en el audit (via `SUBSET_ID_OVERRIDE={'CMAPSS':'FD'}`), por tanto audit y piloto deben coincidir exactamente en `n_canales=24` y `diff_audit == 0`.

El detector del piloto identifica `unit_col='id'`, `target_col='rul'`, `subset_id_col='FD'`, `time_col=None` y deriva `signal_cols` excluyendo todos esos. Si la discrepancia con el audit no es exactamente 0, cell 13 falla con un mensaje explicito (a menos que `ALLOW_CHANNEL_DISCREPANCY=True`, no recomendado).


In [ ]:
df_ref = splits.get('train', next(iter(splits.values())))
cols   = list(df_ref.columns)
print(f'Columnas del DataFrame ({len(cols)}):')
print(cols)
print()
df_ref.head()

In [ ]:
# Detector explicito para CMAPSS. No usamos el del audit aqui: queremos control fino.
unit_col      = 'id'   if 'id' in cols      else None
target_col    = 'rul'  if 'rul' in cols     else None
subset_id_col = 'FD'   if 'FD' in cols      else None  # CMAPSS subset id (FD001-FD004)
time_col      = 'time' if 'time' in cols    else ('cycle' if 'cycle' in cols else None)

# Senal: numericas que no sean unit/subset/target/time
metadata_cols = [c for c in (unit_col, subset_id_col) if c is not None]
excluded_cols = sorted({c for c in (unit_col, subset_id_col, target_col, time_col) if c is not None})
signal_cols   = [c for c in cols
                 if c not in set(excluded_cols) and df_ref[c].dtype.kind in 'fiub']

n_canales_pilot = len(signal_cols)
diff_audit      = n_canales_pilot - (audit_n_canales or 0)

print(f'unit_col       : {unit_col}')
print(f'subset_id_col  : {subset_id_col}')
print(f'time_col       : {time_col}')
print(f'target_col     : {target_col}')
print(f'metadata_cols  : {metadata_cols}')
print(f'excluded_cols  : {excluded_cols}')
print(f'signal_cols ({n_canales_pilot}):')
print(signal_cols)
print()
print(f'audit n_canales={audit_n_canales} | piloto n_canales={n_canales_pilot} | diff={diff_audit}')

# Audit v2.3 ya excluye FD via SUBSET_ID_OVERRIDE -> diff debe ser exactamente 0.
if not ALLOW_CHANNEL_DISCREPANCY:
    assert diff_audit == 0, (
        f'audit y piloto deben coincidir en n_canales con audit v2.3 (SUBSET_ID_OVERRIDE '
        f'incluye CMAPSS->FD). diff={diff_audit}. Pon ALLOW_CHANNEL_DISCREPANCY=True solo '
        f'si la discrepancia es deliberada y documentada.'
    )
print('OK: audit v2.3 y piloto v0.4 coinciden en n_canales (FD excluido como subset_id).')

# --- Asserts de CMAPSS completo (CLAUDE.md sec.4: procesar el dataset entero) ---
subset_ids = sorted({int(v) for df in splits.values() for v in df[subset_id_col].unique()})
print(f'\nSubset IDs encontrados (FDs): {subset_ids}')
assert subset_ids == [1, 2, 3, 4], (
    f'CMAPSS deberia traer los 4 FDs (FD001-FD004). Encontrados: {subset_ids}. '
    f'Si phmd solo entrega un subset, el piloto debe parar.'
)

# Sanity check vs audit: trayectorias = split + FD + unit_id.
# Audit v2.3 agrupa exactamente asi en metricas_forma_volumen, asi que las cifras
# deben coincidir byte a byte. Si no coinciden, hay desalineacion estructural.
audit_n_units = int(fila['n_unidades_total']) if 'n_unidades_total' in fila and not pd.isna(fila['n_unidades_total']) else None
pilot_n_units = sum(df[unit_col].nunique() for df in splits.values())
pilot_n_trajectories_aprox = sum(
    df.groupby([subset_id_col, unit_col]).ngroups for df in splits.values()
) if subset_id_col else pilot_n_units
print(f'Audit n_unidades_total (trayectorias): {audit_n_units}')
print(f'Piloto: unit_col distinct = {pilot_n_units} (sin distinguir FD)')
print(f'Piloto: trayectorias (FD+unit) por split sumadas = {pilot_n_trajectories_aprox}')
if audit_n_units is not None:
    assert pilot_n_trajectories_aprox == audit_n_units, (
        f'trayectorias del piloto ({pilot_n_trajectories_aprox}) difieren del audit ({audit_n_units}). '
        f'Con audit v2.3 deben coincidir exactamente: ambos agrupan por '
        f'(split + subset_id + unit_col).'
    )
    print('OK: trayectorias del piloto == audit (alineacion estructural exacta).')
else:
    print('AVISO: audit_n_units es None en el CSV; no se valida igualdad de trayectorias.')


## 6. Splits originales y `unit_global_id`

`phmd` entrega train/val/test directamente, asi que el split se aplica **antes** del ventaneo (regla obligatoria de CLAUDE.md sec.6). Para evitar colisiones de `unit_id` entre subsets FD001-FD004 y entre splits, construimos:

```
unit_global_id = CMAPSS_FD{FD}_{split_origen}_unit{unit_id}
```

Esto garantiza unicidad absoluta de cada trayectoria a traves de splits y subsets, y es el campo sobre el que se hara el check anti-leakage.

In [ ]:
import numpy as np


def trajectory_id_for(split_name, fd_val, unit_val):
    """Identificador unico de trayectoria procesada (incluye el split).

    Por construccion no puede colisionar entre splits. Garantiza unicidad pero
    no es un check fuerte de leakage por si mismo (ver asset_key_without_split).
    """
    return f'CMAPSS_FD{int(fd_val):03d}_{split_name}_unit{int(unit_val):03d}'


def asset_key_without_split_for(fd_val, unit_val):
    """Identificador de activo fisico SIN el split.

    Esto si puede repetirse entre splits originales del benchmark (en CMAPSS
    train/test reutilizan unit_id dentro de cada FD). En anti_leakage_report se
    documenta que es comportamiento esperado del benchmark, no leakage.
    """
    return f'CMAPSS_FD{int(fd_val):03d}_unit{int(unit_val):03d}'


# -----------------------------------------------------------------------------
# Asignamos trajectory_id PRIMERO en todos los splits que phmd haya entregado
# (tipicamente train y test). Necesitamos trajectory_id antes del fallback de
# val porque el feedback exige que el split se haga por trajectory_id, no por
# unit_col crudo (unit_id colisiona entre FDs en CMAPSS).
# -----------------------------------------------------------------------------
for split_name in list(splits.keys()):
    df = splits[split_name].copy()
    df['trajectory_id'] = df.apply(
        lambda r: trajectory_id_for(split_name, r[subset_id_col], r[unit_col]), axis=1,
    )
    df['asset_key_without_split'] = df.apply(
        lambda r: asset_key_without_split_for(r[subset_id_col], r[unit_col]), axis=1,
    )
    df['unit_global_id'] = df['trajectory_id']   # alias retro-compatible
    splits[split_name] = df

# -----------------------------------------------------------------------------
# Si phmd no entrego val, lo creamos desde train por TRAYECTORIA completa
# (nunca desde test, nunca cortando dentro de una unidad). Usamos trajectory_id
# como llave: garantiza que no haya solape de motor concreto entre train y val
# aunque dos FDs reutilicen un mismo unit_id crudo.
# -----------------------------------------------------------------------------
val_fallback_aplicado = False
val_fallback_meta = None
if 'val' not in splits or splits['val'] is None or len(splits['val']) == 0:
    print('AVISO: phmd no entrego val. Se creara val desde train por trayectoria completa.')
    df_train = splits['train']
    trajs_train = df_train['trajectory_id'].unique()
    rng = np.random.default_rng(42)
    n_val = max(1, int(round(0.10 * len(trajs_train))))
    trajs_val = set(rng.choice(trajs_train, size=n_val, replace=False))
    df_val_new   = df_train[df_train['trajectory_id'].isin(trajs_val)].copy()
    df_train_new = df_train[~df_train['trajectory_id'].isin(trajs_val)].copy()
    # Re-etiquetamos trajectory_id de las trayectorias movidas a val: ahora el
    # split_origen es 'val', no 'train'. Asi el id sigue codificando el split
    # real y el asset_key_without_split se mantiene (fisicamente es el mismo
    # motor). Esto es importante para los checks anti-leakage posteriores.
    def _reasignar_split(df, split_destino):
        df = df.copy()
        df['trajectory_id'] = df.apply(
            lambda r: trajectory_id_for(split_destino, r[subset_id_col], r[unit_col]), axis=1,
        )
        df['unit_global_id'] = df['trajectory_id']
        return df
    df_val_new   = _reasignar_split(df_val_new,   'val')
    df_train_new = _reasignar_split(df_train_new, 'train')
    splits['val']   = df_val_new
    splits['train'] = df_train_new
    val_fallback_aplicado = True
    val_fallback_meta = {
        'aplicado':                True,
        'criterio':                'trajectory_id (no unit_col crudo)',
        'n_trayectorias_movidas':  len(trajs_val),
        'n_trayectorias_train_restante': df_train_new['trajectory_id'].nunique(),
        'seed':                    42,
        'ratio_objetivo':          0.10,
    }
    print(f'  val creada por trajectory_id: {len(trajs_val)} trayectorias movidas. '
          f'train restante: {df_train_new["trajectory_id"].nunique()} trayectorias.')

for split_name, df in splits.items():
    n_unidades = df['trajectory_id'].nunique()
    n_fds      = df[subset_id_col].nunique()
    print(f'  {split_name:5s}: {len(df):>7,d} filas, {n_unidades} trayectorias, '
          f'FDs={sorted(df[subset_id_col].unique())}')

print()
print(f'val_fallback_aplicado: {val_fallback_aplicado}')


## 7. Check anti-leakage previo al ventaneo

Antes de generar ventanas, verificamos que ningun `unit_global_id` aparece en mas de un split. Esto es el check fundamental de leakage (CLAUDE.md sec.6).

In [ ]:
# Check 1: trayectorias unicas (con split en el id). Pasa por construccion del id.
trajs = {sn: set(df['trajectory_id'].unique()) for sn, df in splits.items()}
interseccion_train_val  = trajs.get('train', set()) & trajs.get('val', set())
interseccion_train_test = trajs.get('train', set()) & trajs.get('test', set())
interseccion_val_test   = trajs.get('val', set())   & trajs.get('test', set())

print('CHECK 1: trajectory_id disjunto entre splits (incluye split en el id)')
print(f'  train trajectories: {len(trajs.get("train", set()))}')
print(f'  val   trajectories: {len(trajs.get("val", set()))}')
print(f'  test  trajectories: {len(trajs.get("test", set()))}')
print(f'  interseccion train-val : {len(interseccion_train_val)}')
print(f'  interseccion train-test: {len(interseccion_train_test)}')
print(f'  interseccion val-test  : {len(interseccion_val_test)}')
assert len(interseccion_train_val)  == 0, 'leakage train-val (trajectory_id)'
assert len(interseccion_train_test) == 0, 'leakage train-test (trajectory_id)'
assert len(interseccion_val_test)   == 0, 'leakage val-test (trajectory_id)'
print('  OK')

# Check 2: asset_key_without_split puede repetirse entre splits originales del
# benchmark (CMAPSS train/test reutilizan unit_id dentro de cada FD). Es comportamiento
# esperado del benchmark, no leakage. Lo documentamos explicitamente.
assets = {sn: set(df['asset_key_without_split'].unique()) for sn, df in splits.items()}
asset_reused_train_test = assets.get('train', set()) & assets.get('test', set())
asset_reused_train_val  = assets.get('train', set()) & assets.get('val', set())

print()
print('CHECK 2: raw_unit_ids reusados entre splits originales (informativo, no leakage)')
print(f'  asset_key train-test : {len(asset_reused_train_test)}')
print(f'  asset_key train-val  : {len(asset_reused_train_val)}')
print('  Nota: en CMAPSS, train y test mantienen el mismo numero de unidad para distintas')
print('  trayectorias del mismo motor (por ejemplo unit 1 de train != unit 1 de test).')
print('  El benchmark lo trata como secuencias independientes; el piloto las trata como')
print('  trayectorias distintas via trajectory_id. Documentado como expected_for_CMAPSS.')

## 8. Tratamiento de NaN local por unidad

Solo dentro de cada unidad y solo dentro de cada split. CMAPSS no tiene NaN, asi que esta etapa es no-op aqui, pero queda implementada para coherencia con datasets de calidad menor. Se descarta una unidad si supera `MAX_NAN_PCT_UNIDAD` en algun canal. Se interpola linealmente el resto.

In [ ]:
import numpy as np
import pandas as pd


def limpiar_nans_split(df_split, unit_col, signal_cols, max_nan_pct):
    n_descartadas = 0
    dfs_unidades  = []
    for uid, sub in df_split.groupby('trajectory_id', sort=False):
        X = sub[signal_cols].to_numpy()
        if np.isnan(X).any():
            nan_pct = np.isnan(X).mean(axis=0) * 100
            if (nan_pct > max_nan_pct).any():
                n_descartadas += 1
                continue
            sub = sub.copy()
            sub[signal_cols] = sub[signal_cols].interpolate(
                method='linear', axis=0, limit_direction='both',
            )
        dfs_unidades.append(sub)
    out = pd.concat(dfs_unidades, ignore_index=True) if dfs_unidades else df_split.iloc[0:0]
    return out, n_descartadas


# -----------------------------------------------------------------------------
# Antes de cualquier limpieza/ordenamiento, registramos el orden de aparicion
# de cada fila dentro de su trayectoria. Esto es necesario para preservar el
# orden original cuando no hay time_col: phmd suele entregar las muestras ya
# en orden temporal aunque no exponga la columna 'time'/'cycle' explicitamente.
# Si nos limitasemos a ordenar por trajectory_id sin un tie-breaker estable,
# el sort romperia ese orden de facto.
# -----------------------------------------------------------------------------
for split_name in list(splits.keys()):
    df = splits[split_name].copy()
    df['_source_row_order'] = df.groupby('trajectory_id', sort=False).cumcount()
    splits[split_name] = df

source_rows_por_split = {sn: int(len(splits[sn])) for sn in splits.keys()}
source_rows_total     = int(sum(source_rows_por_split.values()))

# Limpieza local de NaN por unidad. Ya tenemos trajectory_id, asi que el
# groupby funciona correctamente.
unidades_descartadas_total = 0
for split_name in list(splits.keys()):
    out, n_desc = limpiar_nans_split(splits[split_name], unit_col, signal_cols, MAX_NAN_PCT_UNIDAD)
    splits[split_name] = out
    unidades_descartadas_total += n_desc
    print(f'  {split_name:5s}: descartadas {n_desc} unidades por NaN.')

print(f'\nTotal unidades descartadas: {unidades_descartadas_total}')

# -----------------------------------------------------------------------------
# Ordenamiento explicito por (trajectory_id, time_col o _source_row_order)
# antes de ventanear. CLAUDE.md sec.5 insiste en ordenar por unidad/tiempo.
# Si no hay time_col, _source_row_order garantiza estabilidad y preserva el
# orden de phmd. Usamos kind='mergesort' (sort estable) para no romper ningun
# tie-breaker ya existente en filas con el mismo time_col.
# -----------------------------------------------------------------------------
order_cols = ['trajectory_id']
if time_col is not None:
    order_cols.append(time_col)
    ordered_by = 'unit_and_time'
else:
    order_cols.append('_source_row_order')
    ordered_by = 'unit_and_source_order'

for split_name in list(splits.keys()):
    splits[split_name] = (
        splits[split_name]
        .sort_values(order_cols, kind='mergesort')
        .reset_index(drop=True)
    )

# Banderas para los checks anti-leakage y el manifest:
ordered_by_unit_and_cycle        = (time_col is not None)
ordered_by_unit_and_source_order = (time_col is None)

print(f'\nOrdenado por {order_cols} (kind=mergesort).')
print(f'  ordered_by_unit_and_cycle        = {ordered_by_unit_and_cycle}')
print(f'  ordered_by_unit_and_source_order = {ordered_by_unit_and_source_order}')
print(f'  source_rows_total                = {source_rows_total:,d}')


## 9. Helper de ventaneo, normalizacion y patching

Generamos, para cada unidad, todas sus ventanas `W=512, S=256`, las normalizamos por instancia y las convertimos al contrato `(C, N, P)`. Las mascaras `valid_time_mask` y `valid_patch_mask` se calculan de la siguiente forma:

- `valid_time_mask[w]` = True si la posicion `w` es real (no padding).
- `valid_patch_mask[c, n]` = True si el patch `n` tiene al menos una muestra real (deriva de `valid_time_mask` aplicada a cada patch).

Para CMAPSS la mascara es identica por canal porque el padding es temporal puro, pero se guarda en `(C, N)` por coherencia con el contrato y por robustez ante datasets con padding por canal.

In [ ]:
import numpy as np


def ventanas_de_unidad(X_unidad, target_unidad,
                       window_size=WINDOW_SIZE, stride=STRIDE,
                       tail_policy=None):
    """Genera ventanas (X_win, vtm, target_win, w_start, w_end) de una unidad.

    Politica de cola (alineada con audit v2.3):
      - tail_policy='pad' (default v0.4): si T > W y queda resto tras la
        ultima ventana completa, se anade una ventana parcial con padding
        por la derecha. Start = ultima_w_end, end = T, padding hasta W.
        valid_time_mask True solo en las primeras `resto` posiciones.
      - tail_policy='drop': comportamiento historico, descarta el resto.

    X_unidad shape (T, C), target_unidad shape (T,). Devuelve una lista de tuplas:
      X_win: (W, C) float32 con padding por la derecha cuando T < W o pad-tail
      vtm  : (W,) bool, True si la posicion es real
      target_win: ultimo valor valido del target en la ventana
      w_start, w_end: limites reales [w_start, w_end) sobre la unidad
    """
    if tail_policy is None:
        tail_policy = TAIL_POLICY
    T, C = X_unidad.shape
    out = []
    if T <= window_size:
        # Serie corta: 1 ventana con padding al final. Misma logica para
        # tail_policy='pad' y 'drop' (no hay cola que descartar/anadir).
        X = np.zeros((window_size, C), dtype='float32')
        X[:T] = X_unidad.astype('float32')
        vtm = np.zeros(window_size, dtype=bool); vtm[:T] = True
        target_win = target_unidad[T - 1]
        out.append((X, vtm, target_win, 0, int(T)))
    else:
        # Ventanas completas con stride
        for inicio in range(0, T - window_size + 1, stride):
            fin = inicio + window_size
            X = X_unidad[inicio:fin].astype('float32')
            vtm = np.ones(window_size, dtype=bool)
            target_win = target_unidad[fin - 1]
            out.append((X, vtm, target_win, int(inicio), int(fin)))
        # Cola: en pad anadimos ventana parcial extra; en drop la descartamos.
        if tail_policy == 'pad':
            ultima_w_end = ((T - window_size) // stride) * stride + window_size
            if ultima_w_end < T:
                resto = T - ultima_w_end
                X = np.zeros((window_size, C), dtype='float32')
                X[:resto] = X_unidad[ultima_w_end:T].astype('float32')
                vtm = np.zeros(window_size, dtype=bool); vtm[:resto] = True
                target_win = target_unidad[T - 1]
                out.append((X, vtm, target_win, int(ultima_w_end), int(T)))
    return out


def normalizar_instancia(X, vtm, eps=1e-6):
    """Instance normalization por canal ignorando padding.

    Devuelve (X_norm, mean, std_used, canales_constantes_mask).
    - mean, std_used tienen shape (C,) y se calculan antes de normalizar.
    - canales_constantes_mask[c] = True si var<eps (canal constante o casi).
    - Para esos canales se usa std=1.0 para evitar division por cero, pero se
      marca explicitamente en la mascara. std.npy guarda std_used (sustituido).
    """
    W, C = X.shape
    m = vtm.astype('float32')[:, None]
    n_validos = m.sum()
    if n_validos < 2:
        return X * m, np.zeros(C, dtype='float32'), np.ones(C, dtype='float32'), \
               np.zeros(C, dtype=bool)
    mean = ((X * m).sum(axis=0) / n_validos).astype('float32')
    var  = (((X - mean) ** 2 * m).sum(axis=0) / n_validos).astype('float32')
    canales_constantes = (var < eps)
    std_raw = np.sqrt(np.maximum(var, 0.0)).astype('float32')
    std_used = np.where(canales_constantes, 1.0, np.maximum(std_raw, eps)).astype('float32')
    X_norm = (X - mean) / std_used
    X_norm = X_norm * m   # padding queda a 0
    return X_norm.astype('float32'), mean, std_used, canales_constantes


def patches_C_N_P(X_norm, vtm, patch_size=PATCH_SIZE):
    """Convierte (W, C) en (C, N, P) y deriva valid_patch_mask (C, N) bool.

    valid_patch_mask[c, n] = True si el patch n tiene al menos una muestra
    real (deriva de valid_time_mask aplicado por patch). La loss SSL debe
    usar ademas valid_time_mask.reshape(N, P) para enmascarar el padding
    dentro de un patch parcialmente valido (CLAUDE.md sec. 14).
    """
    W, C = X_norm.shape
    assert W % patch_size == 0, f'W={W} no es multiplo de patch_size={patch_size}'
    N = W // patch_size
    patches = X_norm.reshape(N, patch_size, C).transpose(2, 0, 1).astype('float32')
    vtm_patches = vtm.reshape(N, patch_size).any(axis=1)
    valid_patch_mask = np.broadcast_to(vtm_patches, (C, N)).astype(bool).copy()
    return patches, valid_patch_mask


print(f'Helpers de ventaneo, normalizacion y patching definidos. tail_policy default={TAIL_POLICY}.')


## 10. Generar samples por split

Para cada split y cada unidad: extraer ventanas, normalizar (guardando mean/std antes), patchear y empaquetar todo lo necesario por sample. Los samples se mantienen en memoria hasta la escritura a shards. CMAPSS cabe en memoria en el piloto; el numero exacto de ventanas se valida contra audit v2.3 en los asserts de cierre.


In [ ]:
from datetime import datetime, timezone

samples_por_split = {sn: [] for sn in splits.keys()}
warnings_pipeline = []
std_zero_por_canal = {}          # canal -> set(trajectory_id)
_traj_con_std_zero = set()

# Control de filas (v0.4):
# - valid_time_positions_across_windows: suma de vtm.sum() sobre todas las
#   ventanas. Con ventanas solapadas (stride<W) cuenta multiplicidades; NO
#   es equivalente a filas unicas cubiertas.
# - source_rows_covered_unique: filas crudas tocadas por al menos una ventana.
#   Con tail_policy='pad' debe coincidir con source_rows_total.
# - dropped_tail_rows_due_to_windowing: deberia ser 0 con pad.
valid_time_positions_across_windows = 0
dropped_tail_rows_due_to_windowing  = 0

for split_name, df in splits.items():
    for traj_id, sub in df.groupby('trajectory_id', sort=False):
        X_unidad      = sub[signal_cols].to_numpy()
        target_unidad = sub[target_col].to_numpy()
        T_unidad      = X_unidad.shape[0]
        ventanas      = ventanas_de_unidad(X_unidad, target_unidad, tail_policy=TAIL_POLICY)
        if not ventanas:
            continue
        ultimo_w_end = max(int(w_end) for (_, _, _, _, w_end) in ventanas)
        if T_unidad > ultimo_w_end:
            dropped_tail_rows_due_to_windowing += (T_unidad - ultimo_w_end)
        for idx_w, (X_win, vtm, target_win, w_start, w_end) in enumerate(ventanas):
            X_norm, mean_c, std_used, const_mask = normalizar_instancia(X_win, vtm)
            if const_mask.any():
                if traj_id not in _traj_con_std_zero:
                    _traj_con_std_zero.add(traj_id)
                    canales_const = [signal_cols[c] for c in np.where(const_mask)[0]]
                    for nombre_canal in canales_const:
                        std_zero_por_canal.setdefault(nombre_canal, set()).add(traj_id)
                    warnings_pipeline.append({
                        'tipo': 'std_zero', 'trajectory_id': traj_id,
                        'canales_constantes': canales_const,
                        'detalle': 'al menos un canal con var<eps en alguna ventana',
                    })
            patches, vpm = patches_C_N_P(X_norm, vtm)
            valid_time_positions_across_windows += int(vtm.sum())
            sample = {
                'patches':                 patches,
                'valid_time_mask':         vtm,
                'valid_patch_mask':        vpm,
                'mean':                    mean_c,
                'std_used':                std_used,
                'canales_constantes_mask': const_mask,
                'target_window':           float(target_win),
                'target_warning':          'rul_negative_values' if (audit_rul_min is not None and audit_rul_min < 0) else None,
                'trajectory_id':           traj_id,
                'asset_key_without_split': sub['asset_key_without_split'].iloc[0],
                'idx_window':              idx_w,
                'split_origen':            split_name,
                'subset_id':               int(sub[subset_id_col].iloc[0]) if subset_id_col else None,
                'unit_id':                 int(sub[unit_col].iloc[0]),
                'window_start':            w_start,
                'window_end':              w_end,
            }
            samples_por_split[split_name].append(sample)

# source_rows_covered_unique: con pad, todas las filas estan cubiertas por al
# menos una ventana, asi que equivale a source_rows_total. Con drop, las
# filas de cola descartadas no estan cubiertas.
source_rows_covered_unique = source_rows_total - dropped_tail_rows_due_to_windowing

# Resumen agregado de std_zero
_n_trajs_total = int(sum(df['trajectory_id'].nunique() for df in splits.values()))
warnings_aggregated_std_zero = [
    {
        'tipo':                          'std_zero_canal_constante',
        'canal':                         canal,
        'n_trayectorias_afectadas':      len(trajs),
        'porcentaje_trayectorias_total': round(100.0 * len(trajs) / max(1, _n_trajs_total), 2),
    }
    for canal, trajs in sorted(std_zero_por_canal.items(), key=lambda kv: -len(kv[1]))
]

# Assert: con tail_policy=pad, no debe haber cola descartada.
if TAIL_POLICY == 'pad':
    assert dropped_tail_rows_due_to_windowing == 0, (
        f'tail_policy=pad debe garantizar dropped_tail_rows=0, '
        f'pero salio {dropped_tail_rows_due_to_windowing}. Posible bug en ventanas_de_unidad.'
    )

# Backward-compat: nombre antiguo
valid_time_positions_written = valid_time_positions_across_windows

for split_name, lst in samples_por_split.items():
    print(f'  {split_name:5s}: {len(lst):>5d} samples generados.')
print(f'\nWarnings del pipeline (detalle): {len(warnings_pipeline)}')
print(f'Warnings agregados std_zero (por canal): {len(warnings_aggregated_std_zero)}')
for w in warnings_aggregated_std_zero[:5]:
    print(f'  - {w}')
print()
print(f'tail_policy                          = {TAIL_POLICY}')
print(f'valid_time_positions_across_windows  = {valid_time_positions_across_windows:,d}  (cuenta solapes)')
print(f'source_rows_covered_unique           = {source_rows_covered_unique:,d}  (filas crudas unicas cubiertas)')
print(f'dropped_tail_rows_due_to_windowing   = {dropped_tail_rows_due_to_windowing:,d}  (debe ser 0 con pad)')


## 11. Asserts de contrato (shapes, dtypes, separacion target/features)

Verificamos antes de escribir nada:

- `patches` shape `(C, N, P)`, dtype float32.
- `valid_time_mask` shape `(W,)`, dtype bool.
- `valid_patch_mask` shape `(C, N)`, dtype bool.
- `mean`, `std` shape `(C,)`, dtype float32.
- `target` separado del array de patches.
- `target_col` no aparece dentro de `signal_cols`.

In [ ]:
import numpy as np

C = n_canales_pilot
N = N_PATCHES
P = PATCH_SIZE
W = WINDOW_SIZE

primer_sample = None
for lst in samples_por_split.values():
    if lst:
        primer_sample = lst[0]
        break
assert primer_sample is not None, 'no se ha generado ningun sample'

assert primer_sample['patches'].shape == (C, N, P), f'patches shape {primer_sample["patches"].shape} != {(C, N, P)}'
assert primer_sample['patches'].dtype == np.float32
assert primer_sample['valid_time_mask'].shape == (W,)
assert primer_sample['valid_time_mask'].dtype == bool
assert primer_sample['valid_patch_mask'].shape == (C, N)
assert primer_sample['valid_patch_mask'].dtype == bool
assert primer_sample['mean'].shape == (C,)
assert primer_sample['mean'].dtype == np.float32
assert primer_sample['std_used'].shape == (C,)
assert primer_sample['std_used'].dtype == np.float32
assert target_col not in signal_cols, f'target_col {target_col} esta en signal_cols (leakage de target)'
assert primer_sample['window_start'] is not None and primer_sample['window_end'] is not None

print('Contrato verificado:')
print(f'  patches          : {primer_sample["patches"].shape}  {primer_sample["patches"].dtype}')
print(f'  valid_time_mask  : {primer_sample["valid_time_mask"].shape}  {primer_sample["valid_time_mask"].dtype}')
print(f'  valid_patch_mask : {primer_sample["valid_patch_mask"].shape}  {primer_sample["valid_patch_mask"].dtype}')
print(f'  mean             : {primer_sample["mean"].shape}  {primer_sample["mean"].dtype}')
print(f'  std_used         : {primer_sample["std_used"].shape}  {primer_sample["std_used"].dtype}')
print(f'  window_start/end : {primer_sample["window_start"]} / {primer_sample["window_end"]}')
print(f'  target separado  : OK ({target_col} no esta en signal_cols)')

## 12. Dataset y DataLoader de PyTorch

`VentanasCMAPSS` es un `IterableDataset` que lee samples desde una lista en memoria (en el piloto) y genera la `ssl_mask` online por cada `__iter__` (= cada epoch en entrenamiento real). La mascara solo cubre patches validos (`valid_patch_mask=True`) y nunca patches de padding. La loss del modelo futuro solo se calculara donde `valid_patch_mask & ssl_mask`.

Para timesteps de padding dentro de patches **parcialmente** validos (un patch borde puede tener mitad real, mitad padding), la mascara `valid_time_mask` se conserva en el sample. El modelo derivara una mascara `(N, P)` desde `valid_time_mask.reshape(N, P)` para no entrenar reconstruccion sobre las posiciones de padding.

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader


def generar_ssl_mask(valid_patch_mask, mask_ratio, rng):
    C, N = valid_patch_mask.shape
    ssl_mask = np.zeros((C, N), dtype=bool)
    for c in range(C):
        validos = np.where(valid_patch_mask[c])[0]
        if len(validos) == 0:
            continue
        k = max(1, int(round(len(validos) * mask_ratio)))
        elegidos = rng.choice(validos, size=k, replace=False)
        ssl_mask[c, elegidos] = True
    return ssl_mask


class VentanasCMAPSS(IterableDataset):
    def __init__(self, samples, mask_ratio=MASK_RATIO, seed=0):
        super().__init__()
        self.samples    = samples
        self.mask_ratio = mask_ratio
        self.seed       = seed

    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        for s in self.samples:
            ssl_mask = generar_ssl_mask(s['valid_patch_mask'], self.mask_ratio, rng)
            yield {
                'patches':          torch.from_numpy(s['patches']),
                'valid_time_mask':  torch.from_numpy(s['valid_time_mask']),
                'valid_patch_mask': torch.from_numpy(s['valid_patch_mask']),
                'ssl_mask':         torch.from_numpy(ssl_mask),
                'mean':             torch.from_numpy(s['mean']),
                'std':              torch.from_numpy(s['std_used']),
                'target':           torch.tensor(s['target_window'], dtype=torch.float32),
                'meta':             {
                    'trajectory_id':           s['trajectory_id'],
                    'asset_key_without_split': s['asset_key_without_split'],
                    'idx_window':              s['idx_window'],
                    'split_origen':            s['split_origen'],
                    'subset_id':               s['subset_id'],
                    'unit_id':                 s['unit_id'],
                    'window_start':            s['window_start'],
                    'window_end':              s['window_end'],
                    'target_warning':          s['target_warning'],
                },
            }


def collate_pilot(batch):
    return {
        'patches':          torch.stack([b['patches'] for b in batch]),
        'valid_time_mask':  torch.stack([b['valid_time_mask'] for b in batch]),
        'valid_patch_mask': torch.stack([b['valid_patch_mask'] for b in batch]),
        'ssl_mask':         torch.stack([b['ssl_mask'] for b in batch]),
        'mean':             torch.stack([b['mean'] for b in batch]),
        'std':              torch.stack([b['std'] for b in batch]),
        'target':           torch.stack([b['target'] for b in batch]),
        'meta':             [b['meta'] for b in batch],
    }


ds_torch = VentanasCMAPSS(samples_por_split['train'])
dl       = DataLoader(ds_torch, batch_size=8, collate_fn=collate_pilot, num_workers=0)
batch = next(iter(dl))

print('Shapes del batch:')
for k, v in batch.items():
    if k == 'meta':
        print(f'  {k:18s} list of dict, len={len(v)}')
    else:
        print(f'  {k:18s} {tuple(v.shape)}  dtype={v.dtype}')

assert batch['patches'].ndim == 4 and batch['patches'].shape[1:] == (C, N, P)
assert batch['patches'].dtype == torch.float32
assert batch['valid_patch_mask'].dtype == torch.bool
assert batch['valid_time_mask'].dtype == torch.bool
assert batch['ssl_mask'].dtype == torch.bool

# Validacion del mask ratio efectivo (CLAUDE.md: solo sobre patches validos)
valid_total  = int(batch['valid_patch_mask'].sum().item())
masked_total = int(batch['ssl_mask'].sum().item())
effective_mask_ratio = (masked_total / valid_total) if valid_total else 0.0
print(f'\nMask ratio efectivo (batch): {effective_mask_ratio:.4f} (objetivo {MASK_RATIO})')
print(f'  patches validos en el batch: {valid_total}')
print(f'  patches enmascarados      : {masked_total}')
assert abs(effective_mask_ratio - MASK_RATIO) < 0.05, \
    f'mask ratio efectivo {effective_mask_ratio:.3f} se desvia mas de 0.05 del objetivo {MASK_RATIO}'
print('OK: batch (B,C,N,P) y mask ratio efectivo dentro de tolerancia.')

## 13. Persistencia a WebDataset (shards consolidados)

Escribimos shards `.tar` con `webdataset.ShardWriter` (`SHARD_MAXCOUNT=5000`). Cada sample va en su propia entrada del tar, con multiples ficheros internos por sample (patches.npy, valid_time_mask.npy, valid_patch_mask.npy, mean.npy, std.npy, target.json, meta.json). **No** se escribe un fichero suelto por ventana en el filesystem: todo va dentro de los shards consolidados.

La organizacion de carpetas dentro de `processed/CMAPSS/` agrupa shards por split:

```
processed/CMAPSS/
  train/CMAPSS-train-000000.tar, ...
  val/CMAPSS-val-000000.tar, ...
  test/CMAPSS-test-000000.tar, ...
  manifest.json
  anti_leakage_report.json
```

In [ ]:
import io
import json
import webdataset as wds


def serializar_sample(s):
    key = f'{s["split_origen"]}_{s["trajectory_id"]}_w{s["idx_window"]:04d}'
    bufs = {}
    arr_map = {
        'patches':          s['patches'],
        'valid_time_mask':  s['valid_time_mask'],
        'valid_patch_mask': s['valid_patch_mask'],
        'mean':             s['mean'],
        'std':              s['std_used'],   # std.npy guarda std_used (sustituido por 1.0 en canales constantes)
    }
    for nombre, arr in arr_map.items():
        buf = io.BytesIO()
        np.save(buf, arr)
        bufs[f'{nombre}.npy'] = buf.getvalue()
    target_payload = {
        'target_raw':                    s['target_window'],   # sin transformar; identico a target_window en CMAPSS
        'target_window':                 s['target_window'],
        'target_col':                    target_col,
        'target_policy':                 'ultimo_valor_valido',
        'target_warning':                s['target_warning'],
        'target_semantics_unresolved':   True,
    }
    meta_payload = {
        'dataset':                    DATASET,
        'role':                       'TRANSFER_TARGET',
        'evaluation_tier':            'primary',
        'pipeline_version':           PIPELINE_VERSION,
        'subset_id':                  s['subset_id'],
        'split_origen':               s['split_origen'],
        'unit_id':                    s['unit_id'],
        'trajectory_id':              s['trajectory_id'],
        'asset_key_without_split':    s['asset_key_without_split'],
        'idx_window':                 s['idx_window'],
        'window_start':               s['window_start'],
        'window_end':                 s['window_end'],
        'window_size':                WINDOW_SIZE,
        'stride':                     STRIDE,
        'patch_size':                 PATCH_SIZE,
    }
    return {
        '__key__':           key,
        **bufs,
        'target.json':       json.dumps(target_payload).encode('utf-8'),
        'meta.json':         json.dumps(meta_payload).encode('utf-8'),
    }


actual_shards = {}
actual_n_windows = {}
actual_size_mb_por_split = {}
for split_name, lst in samples_por_split.items():
    out_dir = PROCESSED_DIR / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    pattern = str(out_dir / f'{DATASET}-{split_name}-%06d.tar')
    with wds.ShardWriter(pattern, maxcount=SHARD_MAXCOUNT) as sink:
        for s in lst:
            sink.write(serializar_sample(s))
    shards = sorted(out_dir.glob(f'{DATASET}-{split_name}-*.tar'))
    actual_shards[split_name] = [str(p.relative_to(DRIVE_ROOT)) for p in shards]
    actual_n_windows[split_name] = len(lst)
    size_mb = sum(p.stat().st_size for p in shards) / 1024 / 1024
    actual_size_mb_por_split[split_name] = round(size_mb, 2)
    print(f'  {split_name:5s}: {len(shards)} shards, {len(lst)} samples, {size_mb:.1f} MB')

## 14. Validacion de lectura: `np.allclose` con la version en memoria

Abrimos el primer shard de train, leemos el primer sample y comprobamos que coincide byte-a-byte (en `np.allclose`) con el que tenemos en memoria. Esto verifica que la serializacion no introduce perdida y que el codigo de decodificacion del DataLoader sera correcto.

In [ ]:
import io
import json
import numpy as np
import webdataset as wds


def decodificar(sample):
    out = {}
    for arr_name in ('patches', 'valid_time_mask', 'valid_patch_mask', 'mean', 'std'):
        out[arr_name] = np.load(io.BytesIO(sample[f'{arr_name}.npy']))
    out['target'] = json.loads(sample['target.json'])
    out['meta']   = json.loads(sample['meta.json'])
    return out


shards_train = sorted((PROCESSED_DIR / 'train').glob(f'{DATASET}-train-*.tar'))
primer_shard = str(shards_train[0])

primer_disk = next(iter(wds.WebDataset(primer_shard, shardshuffle=False).map(decodificar)))
primer_mem  = samples_por_split['train'][0]

# Mapeo nombre_en_disco -> nombre_en_memoria. std.npy guarda std_used.
DISK_TO_MEM = {
    'patches':          'patches',
    'valid_time_mask':  'valid_time_mask',
    'valid_patch_mask': 'valid_patch_mask',
    'mean':             'mean',
    'std':              'std_used',
}

for arr_name_disk, arr_name_mem in DISK_TO_MEM.items():
    a_disk = primer_disk[arr_name_disk]
    a_mem  = primer_mem[arr_name_mem]
    assert a_disk.shape == a_mem.shape, f'{arr_name_disk} shape: disk {a_disk.shape} vs mem {a_mem.shape}'
    assert a_disk.dtype == a_mem.dtype, f'{arr_name_disk} dtype: disk {a_disk.dtype} vs mem {a_mem.dtype}'
    if a_disk.dtype == bool:
        assert (a_disk == a_mem).all(), f'{arr_name_disk} difiere bool'
    else:
        assert np.allclose(a_disk, a_mem), f'{arr_name_disk} difiere float'

# target_window y trajectory_id (no unit_global_id) deben coincidir
assert primer_disk['target']['target_window'] == primer_mem['target_window']
assert primer_disk['target']['target_raw']    == primer_mem['target_window']
assert primer_disk['meta']['trajectory_id']   == primer_mem['trajectory_id']
assert primer_disk['meta']['asset_key_without_split'] == primer_mem['asset_key_without_split']

print('OK: el primer sample en disco coincide con el que esta en memoria.')
print()
print('Meta del sample:')
print(json.dumps(primer_disk['meta'], indent=2))

## 15. `anti_leakage_report.json`

Resumen formal de los checks anti-leakage exigidos por CLAUDE.md sec.6 y sec.9. Se escribe en `processed/CMAPSS/anti_leakage_report.json`.

In [ ]:
import json

# v0.4: solo CONSTRUIMOS el dict de anti_leakage aqui. La escritura a disco
# se hace en cell 35, despues de los asserts duros contra el audit. Si los
# asserts fallan, ningun artefacto (anti_leakage_report.json, manifest.json,
# processed_summary.csv, done.flag) se sobreescribe.
anti_leakage = {
    'pipeline_version':         PIPELINE_VERSION,
    'dataset':                  DATASET,
    'timestamp':                datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'trajectory_id_policy':     'CMAPSS_FD{FD:03d}_{split_origen}_unit{unit_id:03d}',
    'asset_key_without_split_policy': 'CMAPSS_FD{FD:03d}_unit{unit_id:03d}',
    'checks': {
        'no_trajectory_id_shared_across_splits': {
            'result': 'pass',
            'note':   'El trajectory_id incluye el split, asi que no puede colisionar entre '
                      'splits por construccion. Garantiza unicidad pero no es un test fuerte '
                      'de leakage por si solo. Ver check raw_unit_ids_reused.',
            'n_train_trajectories': len(trajs.get('train', set())),
            'n_val_trajectories':   len(trajs.get('val', set())),
            'n_test_trajectories':  len(trajs.get('test', set())),
            'intersection_train_val':  len(interseccion_train_val),
            'intersection_train_test': len(interseccion_train_test),
            'intersection_val_test':   len(interseccion_val_test),
        },
        'raw_unit_ids_reused_across_original_splits': {
            'result': 'documented_expected_for_CMAPSS',
            'note':   'En CMAPSS los unit_id se reutilizan entre train/test dentro del mismo '
                      'FD (motores distintos comparten numero). El benchmark trata train y '
                      'test como secuencias independientes; el piloto las trata como '
                      'trayectorias distintas via trajectory_id. No es leakage.',
            'asset_key_reused_train_test': len(asset_reused_train_test),
            'asset_key_reused_train_val':  len(asset_reused_train_val),
        },
        'split_applied_before_windowing': {
            'result': 'pass',
            'evidence': 'Splits originales de phmd respetados. Si val no llego de phmd, se '
                        'derivo de train por TRAYECTORIA completa (no por unit_col crudo). '
                        'Ventaneo posterior a la asignacion de split.',
            'val_fallback': val_fallback_meta,
        },
        'data_ordered_by_unit_and_time': {
            'result': ('pass' if ordered_by_unit_and_cycle else 'pass_without_time_col'),
            'order_cols': order_cols,
            'ordered_by_unit_and_cycle':        ordered_by_unit_and_cycle,
            'ordered_by_unit_and_source_order': ordered_by_unit_and_source_order,
            'note': 'Si time_col existe, se ordena por (trajectory_id, time_col); en su '
                    'defecto se usa _source_row_order para preservar el orden de phmd. '
                    'Sort estable (kind=mergesort).',
        },
        'no_test_stats_used_in_normalization': {
            'result': 'pass',
            'evidence': 'instance normalization por ventana y canal; no se usa media/std globales ni de test.',
        },
        'patches_shape_contract': {
            'result': 'pass',
            'shape':  list(primer_sample['patches'].shape),
            'dtype':  str(primer_sample['patches'].dtype),
        },
        'batch_shape_contract': {
            'result': 'pass',
            'shape':  list(batch['patches'].shape),
            'dtype':  str(batch['patches'].dtype),
        },
        'masks_dtype_bool': {
            'result': 'pass',
            'valid_time_mask':  str(batch['valid_time_mask'].dtype),
            'valid_patch_mask': str(batch['valid_patch_mask'].dtype),
            'ssl_mask':         str(batch['ssl_mask'].dtype),
        },
        'target_separated_from_features': {
            'result': 'pass',
            'target_col': target_col,
            'evidence':  f'{target_col} no aparece en signal_cols (asserted) y se guarda en target.json.',
        },
        'effective_mask_ratio_within_tolerance': {
            'result': 'pass',
            'effective_mask_ratio': round(effective_mask_ratio, 4),
            'target_mask_ratio':    MASK_RATIO,
            'tolerance':            0.05,
        },
    },
    'warnings_known_from_audit': warnings_conocidos,
    'warnings_from_pipeline':    warnings_pipeline,
    'warnings_aggregated_std_zero': warnings_aggregated_std_zero,
}

# NO se escribe a disco aqui. Se valida primero el resto contra audit (cell 35)
# y solo si pasa se escriben los artefactos.
results_ok = ('pass', 'pass_without_time_col', 'documented_expected_for_CMAPSS')
all_pass = all(c['result'] in results_ok for c in anti_leakage['checks'].values())
print(f'anti_leakage dict construido (no escrito todavia).')
print(f'Resultado global: {"PASS" if all_pass else "FAIL"}')
for nombre, info in anti_leakage['checks'].items():
    print(f'  {nombre:50s} {info["result"]}')


## 16. Manifest CMAPSS (calculado desde los shards escritos)

El manifest **no copia metricas del audit**. Se calcula desde lo que realmente se ha escrito: numero de samples por split, numero de canales efectivos, padding ratio real, etc. Despues compara con los valores estimados en el audit y registra cualquier diferencia.

In [ ]:
import hashlib
import json
import math
import os
import subprocess
from collections import defaultdict

# ----------------------------------------------------------------------------
# 1. METRICAS reales del piloto
# ----------------------------------------------------------------------------
n_windows_actual_por_split = {sn: len(lst) for sn, lst in samples_por_split.items()}
n_windows_actual_total     = sum(n_windows_actual_por_split.values())

n_patches_por_split        = {sn: len(lst) * (WINDOW_SIZE // PATCH_SIZE) for sn, lst in samples_por_split.items()}
n_valid_patches_por_split  = {sn: sum(int(s['valid_patch_mask'][0].sum()) for s in lst)
                              for sn, lst in samples_por_split.items()}
n_temporal_patches_actual  = sum(n_valid_patches_por_split.values())
n_channel_patches_actual   = sum(
    int(s['valid_patch_mask'].sum()) for lst in samples_por_split.values() for s in lst
)

total_muestras_padding     = sum(
    int((~s['valid_time_mask']).sum())
    for lst in samples_por_split.values() for s in lst
)
padding_ratio_actual = total_muestras_padding / (n_windows_actual_total * WINDOW_SIZE) if n_windows_actual_total else 0.0

actual_n_shards      = sum(len(shs) for shs in actual_shards.values())
actual_size_mb_total = round(sum(actual_size_mb_por_split.values()), 2)

n_trajectories_actual = {sn: len(set(s['trajectory_id'] for s in lst))
                          for sn, lst in samples_por_split.items()}
n_assets_actual = {sn: len(set(s['asset_key_without_split'] for s in lst))
                   for sn, lst in samples_por_split.items()}

# Metricas densas alineadas con audit v2.3
pilot_dense_temporal_patches = n_windows_actual_total * N_PATCHES
pilot_dense_channel_patches  = pilot_dense_temporal_patches * n_canales_pilot
audit_dense_channel_patches  = (audit_n_windows * audit_n_canales * N_PATCHES
                                if audit_n_windows and audit_n_canales else None)
invalid_patch_ratio = (
    round(1.0 - (n_channel_patches_actual / pilot_dense_channel_patches), 4)
    if pilot_dense_channel_patches else None
)
dense_vs_valid_ratio = (
    round(pilot_dense_channel_patches / n_channel_patches_actual, 4)
    if n_channel_patches_actual else None
)


# ----------------------------------------------------------------------------
# 2. pipeline_code_version (env var -> git rev-parse read-only -> None)
# ----------------------------------------------------------------------------
def _get_pipeline_code_version():
    v = os.environ.get('PHMD_PIPELINE_CODE_VERSION') or os.environ.get('GIT_COMMIT')
    if v:
        return v
    try:
        out = subprocess.check_output(
            ['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
            stderr=subprocess.DEVNULL, timeout=5,
        ).decode().strip()
        return out or None
    except Exception:
        return None

pipeline_code_version = _get_pipeline_code_version()


# ----------------------------------------------------------------------------
# 3. pipeline_config_hash completo
# ----------------------------------------------------------------------------
def _hash_lista(items):
    """Hash sensible al orden: el orden de signal_cols define el orden de
    canales en el contrato (C, N, P) y debe formar parte de la trazabilidad."""
    return hashlib.sha256(
        json.dumps(list(items), ensure_ascii=False).encode('utf-8')
    ).hexdigest()[:16]

signal_cols_hash   = _hash_lista(signal_cols)
metadata_cols_hash = _hash_lista(metadata_cols)

PIPELINE_CONFIG = {
    'dataset':              DATASET,
    'role':                 'TRANSFER_TARGET',
    'audit_version':        audit_version,
    'pipeline_version':     PIPELINE_VERSION,
    'tail_policy':          TAIL_POLICY,
    'window_size':          WINDOW_SIZE,
    'stride':               STRIDE,
    'patch_size':           PATCH_SIZE,
    'mask_ratio':           MASK_RATIO,
    'shard_maxcount':       SHARD_MAXCOUNT,
    'normalization_policy': 'instance_norm_por_ventana_y_canal_ignorando_padding',
    'nan_policy':           f'interpolacion_linear_por_unidad_descarte_si_pct_nan>{MAX_NAN_PCT_UNIDAD}',
    'target_policy':        'ultimo_valor_valido',
    'target_col':           target_col,
    'subset_id_col':        subset_id_col,
    'signal_cols_hash':     signal_cols_hash,
    'metadata_cols_hash':   metadata_cols_hash,
}
pipeline_config_str  = json.dumps(PIPELINE_CONFIG, sort_keys=True)
pipeline_config_hash = hashlib.sha256(pipeline_config_str.encode('utf-8')).hexdigest()[:16]


# ----------------------------------------------------------------------------
# 4. ASSERTS DUROS contra el audit v2.3.
# Antes de escribir ningun artefacto final, validamos que el piloto coincide
# byte a byte con el audit en las metricas comparables. Si algo falla,
# abortamos aqui: no se escribe anti_leakage_report, manifest, processed_summary
# ni done.flag.
# ----------------------------------------------------------------------------
print()
print('--- Asserts duros v0.4 contra audit v2.3 (antes de escribir artefactos) ---')

assert audit_version == '2.3', f'audit_version={audit_version}, se requiere 2.3'
assert audit_tail_policy == TAIL_POLICY == 'pad', (
    f'tail_policy mismatch: audit={audit_tail_policy}, piloto={TAIL_POLICY}'
)
print(f'  audit_version=2.3 y tail_policy=pad alineados  OK')

assert n_windows_actual_total == audit_n_windows, (
    f'n_windows piloto={n_windows_actual_total} vs audit={audit_n_windows}'
)
print(f'  n_windows: piloto={n_windows_actual_total} == audit={audit_n_windows}')

assert round(padding_ratio_actual, 4) == round(audit_pad_ratio, 4), (
    f'padding piloto={padding_ratio_actual} vs audit={audit_pad_ratio}'
)
print(f'  padding_ratio: piloto={round(padding_ratio_actual,4)} == audit={round(audit_pad_ratio,4)}')

assert n_temporal_patches_actual == audit_n_temporal_patches, (
    f'temporal_patches piloto={n_temporal_patches_actual} vs audit={audit_n_temporal_patches}'
)
print(f'  n_temporal_patches: piloto={n_temporal_patches_actual} == audit={audit_n_temporal_patches}')

assert n_channel_patches_actual == audit_n_channel_patches_valid, (
    f'channel_patches piloto={n_channel_patches_actual} vs audit={audit_n_channel_patches_valid}'
)
print(f'  n_channel_patches: piloto={n_channel_patches_actual} == audit={audit_n_channel_patches_valid}')

if audit_n_dense_temporal_patches is not None:
    assert pilot_dense_temporal_patches == audit_n_dense_temporal_patches, (
        f'dense_temporal piloto={pilot_dense_temporal_patches} vs audit={audit_n_dense_temporal_patches}'
    )
    print(f'  n_dense_temporal_patches: piloto={pilot_dense_temporal_patches} == audit={audit_n_dense_temporal_patches}')

if audit_n_dense_channel_patches is not None:
    assert pilot_dense_channel_patches == audit_n_dense_channel_patches, (
        f'dense_channel piloto={pilot_dense_channel_patches} vs audit={audit_n_dense_channel_patches}'
    )
    print(f'  n_dense_channel_patches: piloto={pilot_dense_channel_patches} == audit={audit_n_dense_channel_patches}')

# Ratios: redondeo tolerante (audit redondea a 4 decimales)
if audit_invalid_patch_ratio is not None and invalid_patch_ratio is not None:
    assert abs(invalid_patch_ratio - audit_invalid_patch_ratio) < 1e-3, (
        f'invalid_patch_ratio piloto={invalid_patch_ratio} vs audit={audit_invalid_patch_ratio}'
    )
    print(f'  invalid_patch_ratio: piloto={invalid_patch_ratio} ~= audit={audit_invalid_patch_ratio}')

if audit_dense_vs_valid_ratio is not None and dense_vs_valid_ratio is not None:
    assert abs(dense_vs_valid_ratio - audit_dense_vs_valid_ratio) < 1e-3, (
        f'dense_vs_valid_ratio piloto={dense_vs_valid_ratio} vs audit={audit_dense_vs_valid_ratio}'
    )
    print(f'  dense_vs_valid_ratio: piloto={dense_vs_valid_ratio} ~= audit={audit_dense_vs_valid_ratio}')

assert audit_estimated_n_shards == actual_n_shards, (
    f'shards: audit_csv={audit_estimated_n_shards} vs actual={actual_n_shards}.'
)
print(f'  estimated_n_shards: audit={audit_estimated_n_shards} == actual={actual_n_shards}')

# Filas crudas: debe coincidir exactamente. El audit cuenta sum(len(df)) sobre
# splits que phmd entrega; el piloto registra lo mismo en source_rows_total.
if audit_n_filas_total is not None:
    assert source_rows_total == audit_n_filas_total, (
        f'source_rows_total piloto={source_rows_total} vs audit_n_filas_total={audit_n_filas_total}. '
        f'Esto indica desalineacion en la lectura de phmd o en el conteo de filas.'
    )
    print(f'  source_rows_total: piloto={source_rows_total} == audit={audit_n_filas_total}')

if TAIL_POLICY == 'pad':
    assert dropped_tail_rows_due_to_windowing == 0, (
        f'tail_policy=pad debe garantizar dropped=0, pero salio {dropped_tail_rows_due_to_windowing}'
    )
    print(f'  dropped_tail_rows: 0 (pad)')
    assert source_rows_covered_unique == source_rows_total, (
        f'source_rows_covered_unique={source_rows_covered_unique} vs source_rows_total={source_rows_total}'
    )
    print(f'  source_rows_covered_unique == source_rows_total ({source_rows_total})')

print('--- Asserts duros PASS. Procedemos a escribir artefactos. ---')
print()


# ----------------------------------------------------------------------------
# 5. ESCRITURA ATOMICA de artefactos (en orden):
#    - anti_leakage_report.json
#    - manifest.json
#    - warnings_detail.jsonl
#    (processed_summary.csv -> cell 37; done.flag -> cell 39)
# ----------------------------------------------------------------------------

# 5a. anti_leakage_report.json
anti_path = PROCESSED_DIR / 'anti_leakage_report.json'
tmp_anti = anti_path.with_suffix('.json.tmp')
tmp_anti.write_text(json.dumps(anti_leakage, indent=2, ensure_ascii=False))
tmp_anti.replace(anti_path)
print(f'anti_leakage_report.json guardado: {anti_path}')


# 5b. manifest.json
diff_audit_actual = {
    'audit_version':                 audit_version,
    'audit_tail_policy':             audit_tail_policy,
    'pipeline_tail_policy':          TAIL_POLICY,
    'audit_n_windows':               audit_n_windows,
    'pilot_n_windows_actual':        n_windows_actual_total,
    'audit_padding_ratio':           audit_pad_ratio,
    'pilot_padding_ratio_actual':    round(padding_ratio_actual, 4),
    'audit_n_canales':               audit_n_canales,
    'pilot_n_canales':               n_canales_pilot,
    'pilot_n_canales_diff_audit':    diff_audit,
    'audit_n_channel_patches_valid': audit_n_channel_patches_valid,
    'pilot_n_channel_patches_valid': n_channel_patches_actual,
    'audit_n_dense_channel_patches': audit_n_dense_channel_patches,
    'pilot_n_dense_channel_patches': pilot_dense_channel_patches,
    'audit_n_temporal_patches':      audit_n_temporal_patches,
    'pilot_n_temporal_patches':      n_temporal_patches_actual,
    'audit_invalid_patch_ratio':     audit_invalid_patch_ratio,
    'pilot_invalid_patch_ratio':     invalid_patch_ratio,
    'audit_dense_vs_valid_ratio':    audit_dense_vs_valid_ratio,
    'pilot_dense_vs_valid_ratio':    dense_vs_valid_ratio,
    'audit_estimated_n_shards':      audit_estimated_n_shards,
    'pilot_actual_n_shards':         actual_n_shards,
    'source_rows_total':             source_rows_total,
    'valid_time_positions_across_windows': valid_time_positions_across_windows,
    'source_rows_covered_unique':    source_rows_covered_unique,
    'dropped_tail_rows_due_to_windowing': dropped_tail_rows_due_to_windowing,
    'audit_n_filas_total':           audit_n_filas_total,
    'val_fallback_aplicado':         val_fallback_aplicado,
    'explicacion_diffs': (
        'Con audit v2.3 (tail_policy=pad) y piloto v0.4 (tail_policy=pad), las '
        'metricas comparables coinciden exactamente: ambos agrupan por '
        'trajectory_id, cuentan patches validos por ventana y aplican la misma '
        'politica de cola. ' + (
            'val_fallback_aplicado=False: phmd entrega train/val/test directamente.'
            if not val_fallback_aplicado else
            'val_fallback_aplicado=True: el piloto redistribuye ~10% de train a val '
            'porque phmd no entregaba val.'
        )
    ),
}

manifest = {
    # Identidad y trazabilidad
    'dataset':                  DATASET,
    'audit_version':            audit_version,
    'pipeline_version':         PIPELINE_VERSION,
    'pipeline_code_version':    pipeline_code_version,
    'pipeline_config_hash':     pipeline_config_hash,
    'pipeline_config':          PIPELINE_CONFIG,
    'tail_policy':              TAIL_POLICY,

    # Role y semantica
    'role':                     'TRANSFER_TARGET',
    'evaluation_tier':          'primary',
    'client':                   None,
    'source_path':              str(RAW_DIR / 'datasets'),
    'shard_paths':              actual_shards,

    # Splits y orden
    'split_policy':             'splits originales de phmd; val creada desde train por trajectory_id si phmd no la entregaba',
    'val_fallback':             val_fallback_meta,
    'time_info': {
        'time_col':                         time_col,
        'order_col':                        (time_col if time_col else '_source_row_order'),
        'ordered_by_unit_and_cycle':        ordered_by_unit_and_cycle,
        'ordered_by_unit_and_source_order': ordered_by_unit_and_source_order,
        'sort_kind':                        'mergesort',
        'sampling_rate_hz':                 None,
        'window_time_seconds':              None,
        'note': 'CMAPSS no expone sampling_rate fiable. W=512 son muestras, no duracion fisica.',
    },

    # Hiperparametros
    'subset_ids':           subset_ids,
    'window_size':          WINDOW_SIZE,
    'stride':               STRIDE,
    'patch_size':           PATCH_SIZE,
    'n_patches':            N_PATCHES,
    'mask_ratio':           MASK_RATIO,
    'effective_mask_ratio_check_batch': round(effective_mask_ratio, 4),
    'batching_policy':      'por_dataset',
    'shard_maxcount':       SHARD_MAXCOUNT,

    # Columnas
    'feature_cols':         signal_cols,
    'metadata_cols':        metadata_cols,
    'excluded_cols':        excluded_cols,
    'target_col':           target_col,
    'target_candidates':    audit_cands_raw.split(';') if audit_cands_raw else [],
    'target_policy':        'ultimo_valor_valido',
    'target_warning':       'rul_negative_values' if (audit_rul_min is not None and audit_rul_min < 0) else None,
    'target_semantics_unresolved': True,

    # Normalizacion
    'normalization_policy':       'instance_norm_por_ventana_y_canal_ignorando_padding',
    'normalization_stats_saved':  True,
    'normalization_std_note':     'std.npy guarda std_used: igual a sqrt(var) salvo en canales constantes (var<eps), donde se sustituye por 1.0.',

    'dtype':                'float32',
    'unit_global_id_policy': ('trajectory_id = CMAPSS_FD{FD:03d}_{split_origen}_unit{unit_id:03d}; '
                              'asset_key_without_split = CMAPSS_FD{FD:03d}_unit{unit_id:03d}'),

    # Conteos por split
    'n_channels':           n_canales_pilot,
    'n_units_por_split':    n_trajectories_actual,
    'n_assets_por_split':   n_assets_actual,
    'n_windows_por_split':  n_windows_actual_por_split,
    'n_windows_total':      n_windows_actual_total,
    'n_patches_por_split':  n_patches_por_split,
    'n_valid_patches_por_split': n_valid_patches_por_split,

    # Patches: validos + densos
    'n_temporal_patches':       n_temporal_patches_actual,
    'n_channel_patches':        n_channel_patches_actual,
    'n_dense_temporal_patches': pilot_dense_temporal_patches,
    'n_dense_channel_patches':  pilot_dense_channel_patches,
    'invalid_patch_ratio':      invalid_patch_ratio,
    'dense_vs_valid_ratio':     dense_vs_valid_ratio,

    # Padding y cola
    'padding_ratio':        round(padding_ratio_actual, 4),
    'row_accounting': {
        'source_rows_total':                  source_rows_total,
        'source_rows_por_split':              source_rows_por_split,
        'source_rows_covered_unique':         source_rows_covered_unique,
        'valid_time_positions_across_windows': valid_time_positions_across_windows,
        'dropped_tail_rows_due_to_windowing': dropped_tail_rows_due_to_windowing,
        'audit_n_filas_total':                audit_n_filas_total,
        'note': ('Con tail_policy=pad, dropped_tail_rows=0 y '
                 'source_rows_covered_unique==source_rows_total. '
                 'valid_time_positions_across_windows cuenta solapes.'),
    },

    # Shards
    'estimated_n_shards_audit': audit_estimated_n_shards,
    'actual_n_shards':          actual_n_shards,
    'actual_size_mb_por_split': actual_size_mb_por_split,
    'actual_size_mb_total':     actual_size_mb_total,

    # Sampling policy (referenciada del audit; piloto no aplica sampler)
    'sampling_policy_referenced_from_audit_summary': True,

    # Anti-leakage y warnings
    'anti_leakage_checks':      {k: v['result'] for k, v in anti_leakage['checks'].items()},
    'warnings':                 warnings_conocidos + warnings_pipeline,
    'warnings_aggregated': {
        'std_zero_por_canal': warnings_aggregated_std_zero,
    },
    'diff_audit_vs_actual': diff_audit_actual,

    'fecha':                datetime.now(timezone.utc).isoformat(timespec='seconds'),
    # NOTA: 'done' eliminado del manifest. El marcador de cierre completo es
    # done.flag (escrito al ULTIMO en cell 39 tras todos los asserts y
    # verificacion de artefactos). Asi `done=True` no aparece prematuramente.
}

manifest_path = PROCESSED_DIR / 'manifest.json'
tmp_manifest = manifest_path.with_suffix('.json.tmp')
tmp_manifest.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))
tmp_manifest.replace(manifest_path)
print(f'manifest.json guardado (atomico): {manifest_path}')
print(f'  pipeline_version       : {PIPELINE_VERSION}')
print(f'  pipeline_code_version  : {pipeline_code_version}')
print(f'  pipeline_config_hash   : {pipeline_config_hash}')
print(f'  audit_version          : {audit_version}')
print(f'  tail_policy            : {TAIL_POLICY}')


# 5c. warnings_detail.jsonl
warnings_detail_path = PROCESSED_DIR / 'warnings_detail.jsonl'
tmp_warnings = warnings_detail_path.with_suffix('.jsonl.tmp')
with tmp_warnings.open('w', encoding='utf-8') as f:
    for w in warnings_pipeline:
        f.write(json.dumps(w, ensure_ascii=False) + '\n')
tmp_warnings.replace(warnings_detail_path)
print(f'warnings_detail.jsonl guardado: {warnings_detail_path} ({len(warnings_pipeline)} entradas)')

# NOTA: done.flag NO se escribe aqui. Se escribe el ultimo en cell 39, despues
# de que processed_summary.csv tambien este escrito.
print()
print('Diff con audit:')
for k, v in diff_audit_actual.items():
    print(f'  {k:38s} {v}')


## 17. `results/processed_summary.csv` — fila CMAPSS

Una fila por dataset procesado, con los numeros reales del piloto. Si el CSV ya existe, se actualiza la fila CMAPSS (no se duplican).

In [ ]:
import pandas as pd

row_cmapss = {
    # Identidad y trazabilidad
    'dataset':              DATASET,
    'role':                 'TRANSFER_TARGET',
    'evaluation_tier':      'primary',
    'audit_version':        audit_version,
    'pipeline_version':     PIPELINE_VERSION,
    'pipeline_code_version': pipeline_code_version,
    'pipeline_config_hash': pipeline_config_hash,
    'tail_policy':          TAIL_POLICY,

    # Hiperparametros
    'window_size':          WINDOW_SIZE,
    'stride':               STRIDE,
    'patch_size':           PATCH_SIZE,
    'mask_ratio':           MASK_RATIO,

    # Estructura
    'n_channels':           n_canales_pilot,
    'subset_ids':           ','.join(str(x) for x in subset_ids),
    'n_trajectories_train': n_trajectories_actual.get('train', 0),
    'n_trajectories_val':   n_trajectories_actual.get('val', 0),
    'n_trajectories_test':  n_trajectories_actual.get('test', 0),

    # Ventanas y patches
    'n_windows_train':      n_windows_actual_por_split.get('train', 0),
    'n_windows_val':        n_windows_actual_por_split.get('val', 0),
    'n_windows_test':       n_windows_actual_por_split.get('test', 0),
    'n_windows_total':      n_windows_actual_total,
    'n_temporal_patches':   n_temporal_patches_actual,
    'n_channel_patches':    n_channel_patches_actual,
    'n_dense_temporal_patches': pilot_dense_temporal_patches,
    'n_dense_channel_patches':  pilot_dense_channel_patches,
    'invalid_patch_ratio':  invalid_patch_ratio,
    'dense_vs_valid_ratio': dense_vs_valid_ratio,
    'padding_ratio':        round(padding_ratio_actual, 4),

    # Row accounting
    'source_rows_total':                  source_rows_total,
    'source_rows_covered_unique':         source_rows_covered_unique,
    'valid_time_positions_across_windows': valid_time_positions_across_windows,
    'dropped_tail_rows_due_to_windowing': dropped_tail_rows_due_to_windowing,

    # Comparacion con audit
    'estimated_n_shards_audit': audit_estimated_n_shards,
    'actual_n_shards':          actual_n_shards,
    'actual_size_mb_total':     actual_size_mb_total,

    # Mask ratio
    'effective_mask_ratio_check_batch': round(effective_mask_ratio, 4),

    # Target
    'target_col':           target_col,
    'target_policy':        'ultimo_valor_valido',
    'target_warning':       manifest['target_warning'],

    # Fecha
    'fecha':                manifest['fecha'],
}

if PROCESSED_SUMMARY.exists():
    df_prev = pd.read_csv(PROCESSED_SUMMARY)
    df_prev = df_prev[df_prev['dataset'] != DATASET]
    df_new  = pd.concat([df_prev, pd.DataFrame([row_cmapss])], ignore_index=True)
else:
    df_new = pd.DataFrame([row_cmapss])
df_new = df_new.sort_values('dataset').reset_index(drop=True)
PROCESSED_SUMMARY.parent.mkdir(parents=True, exist_ok=True)

# Escritura atomica
tmp_summary = PROCESSED_SUMMARY.with_suffix('.csv.tmp')
df_new.to_csv(tmp_summary, index=False)
tmp_summary.replace(PROCESSED_SUMMARY)
print(f'processed_summary.csv actualizado (atomico): {PROCESSED_SUMMARY} ({len(df_new)} filas)')
df_new


## 18. Resumen final

Snapshot de lo realmente hecho: shapes, conteos, warnings, paths. Si todo coincide con lo esperado, podemos proceder a `05_harmonization_full.ipynb` para escalar a los 47 datasets restantes.

In [ ]:
# ----------------------------------------------------------------------------
# Resumen final v0.4. Los asserts criticos ya pasaron en cell 35 (antes de
# escribir manifest.json y processed_summary.csv). Aqui los repetimos como
# duplicado informativo, y SOLO si todo pasa escribimos done.flag al final.
# ----------------------------------------------------------------------------
print(f'=== Piloto v{PIPELINE_VERSION} sobre {DATASET} (tail_policy={TAIL_POLICY}) ===')
print()
print('Shapes y dtypes:')
print(f'  patches          : {primer_sample["patches"].shape}  {primer_sample["patches"].dtype}')
print(f'  valid_time_mask  : {primer_sample["valid_time_mask"].shape}  {primer_sample["valid_time_mask"].dtype}')
print(f'  valid_patch_mask : {primer_sample["valid_patch_mask"].shape}  {primer_sample["valid_patch_mask"].dtype}')
print(f'  mean             : {primer_sample["mean"].shape}  {primer_sample["mean"].dtype}')
print(f'  std_used         : {primer_sample["std_used"].shape}  {primer_sample["std_used"].dtype}')
print(f'  batch patches    : {tuple(batch["patches"].shape)}  {batch["patches"].dtype}')
print()
print('Conteos:')
print(f'  channels (signal): {n_canales_pilot} (audit: {audit_n_canales})')
print(f'  subset_ids (FD)  : {subset_ids}')
print(f'  trayectorias train/val/test: '
      f'{n_trajectories_actual.get("train",0)}/'
      f'{n_trajectories_actual.get("val",0)}/'
      f'{n_trajectories_actual.get("test",0)}')
print(f'  windows train/val/test    : '
      f'{n_windows_actual_por_split.get("train",0)}/'
      f'{n_windows_actual_por_split.get("val",0)}/'
      f'{n_windows_actual_por_split.get("test",0)}')
print(f'  windows total    : {n_windows_actual_total} (audit: {audit_n_windows})')
print(f'  temporal patches : {n_temporal_patches_actual:,d} (audit valid: {audit_n_temporal_patches})')
print(f'  channel patches  : {n_channel_patches_actual:,d} '
      f'(audit valid: {audit_n_channel_patches_valid}; pilot dense: {pilot_dense_channel_patches:,d})')
print(f'  padding ratio    : {padding_ratio_actual:.4f} (audit: {audit_pad_ratio})')
print(f'  invalid_patch_ratio (piloto)  : {invalid_patch_ratio} (audit: {audit_invalid_patch_ratio})')
print(f'  dense_vs_valid_ratio (piloto) : {dense_vs_valid_ratio} (audit: {audit_dense_vs_valid_ratio})')
print(f'  shards (actual) : {actual_n_shards} (audit estim: {audit_estimated_n_shards})')
print(f'  size MB total   : {actual_size_mb_total}')
print(f'  mask ratio efectivo (batch): {effective_mask_ratio:.4f} (objetivo {MASK_RATIO})')
print()
print('Row accounting:')
print(f'  source_rows_total                    : {source_rows_total:,d} (audit: {audit_n_filas_total})')
print(f'  source_rows_covered_unique           : {source_rows_covered_unique:,d}')
print(f'  valid_time_positions_across_windows  : {valid_time_positions_across_windows:,d}')
print(f'  dropped_tail_rows_due_to_windowing   : {dropped_tail_rows_due_to_windowing:,d}')
print()
print('Warnings conocidos del audit:')
for w in warnings_conocidos:
    print(f'  - {w}')
print()
print(f'Warnings del pipeline: {len(warnings_pipeline)} entradas individuales '
      f'(detalle en {PROCESSED_DIR / "warnings_detail.jsonl"}).')
print('Warnings agregados std_zero (por canal):')
for w in warnings_aggregated_std_zero:
    print(f'  - canal={w["canal"]:24s} trayectorias_afectadas={w["n_trayectorias_afectadas"]:>5d} '
          f'({w["porcentaje_trayectorias_total"]}% del total)')
print()
print('Anti-leakage checks:')
for nombre, info in anti_leakage['checks'].items():
    print(f'  {nombre:50s} {info["result"]}')
print()

# ----------------------------------------------------------------------------
# Asserts finales (duplicado informativo). Si llegamos aqui es porque cell 35
# ya valido contra el audit. Verificamos ademas que los artefactos esperados
# existen en disco antes de escribir done.flag.
# ----------------------------------------------------------------------------
print('--- Asserts finales v0.4 (informativos + verificacion de artefactos) ---')
assert audit_version == '2.3', f'audit_version={audit_version}'
assert audit_tail_policy == TAIL_POLICY == 'pad'
assert n_windows_actual_total == audit_n_windows
assert round(padding_ratio_actual, 4) == round(audit_pad_ratio, 4)
assert n_temporal_patches_actual == audit_n_temporal_patches
assert n_channel_patches_actual == audit_n_channel_patches_valid
assert audit_estimated_n_shards == actual_n_shards
if TAIL_POLICY == 'pad':
    assert dropped_tail_rows_due_to_windowing == 0
assert manifest['tail_policy'] == 'pad'
assert manifest['audit_version'] == '2.3'
assert manifest['pipeline_version'] == '0.4'
assert manifest['pipeline_config_hash'] == pipeline_config_hash

# Verificar que los artefactos esperados existen en disco
for art in [PROCESSED_DIR / 'manifest.json',
            PROCESSED_DIR / 'anti_leakage_report.json',
            PROCESSED_DIR / 'warnings_detail.jsonl',
            PROCESSED_SUMMARY]:
    assert art.exists(), f'Artefacto esperado no existe: {art}'
print(f'  Todos los artefactos en disco. OK')

# ----------------------------------------------------------------------------
# done.flag al ULTIMO. Si cualquier assert anterior falla, done.flag no se
# escribe y la reentrancia detectara el dataset como stale.
# ----------------------------------------------------------------------------
done_flag = PROCESSED_DIR / 'done.flag'
done_flag.write_text(PIPELINE_VERSION + '\n')
print(f'  done.flag escrito (ULTIMO): {done_flag}')
print('--- PASS: piloto v0.4 cerrado. ---')
print()

print('Outputs en Drive:')
print(f'  {PROCESSED_DIR}/train/*.tar')
print(f'  {PROCESSED_DIR}/val/*.tar')
print(f'  {PROCESSED_DIR}/test/*.tar')
print(f'  {PROCESSED_DIR}/manifest.json')
print(f'  {PROCESSED_DIR}/anti_leakage_report.json')
print(f'  {PROCESSED_DIR}/warnings_detail.jsonl')
print(f'  {PROCESSED_DIR}/done.flag')
print()
print('Output en repo:')
print(f'  {PROCESSED_SUMMARY}')
print()
print('Politica: este notebook NO ejecuta commit ni push. pipeline_code_version '
      'se calcula con env var (PHMD_PIPELINE_CODE_VERSION o GIT_COMMIT) o '
      'git rev-parse de solo lectura; nunca git add/commit/push.')
